***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 下一节： [8.1 作为最小二乘问题的校准](8_1_calibration_least_squares_problem.ipynb)

***


导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=3, suppress=True)

RNG = np.random.default_rng(20260419)


校准是把“被系统响应污染的数据”重新映射回“尽可能接近天空真值的数据”的过程。若把某一时刻、某一基线上的可见度写成

$$
V_{pq}^{\rm obs} = G_p\,X_{pq}\,G_q^\ast + N_{pq},
$$

那么校准的核心问题就是：在不直接观测到真实天空相干量 $X_{pq}$ 的前提下，如何稳健地估计天线增益 $G_p$、评估模型误差，并判断哪些误差能够被“吸收”为方向无关项，哪些已经必须提升到方向依赖处理。

这也是第 8 章与前面第 5 至第 7 章最关键的衔接点。前面几章告诉我们误差从哪里来；本章则讨论这些误差如何进入求解器、如何影响残差，以及为什么“求得一个解”并不等于“得到可信的科学结果”。


***


## 第 8 章：校准，从误差模型到可操作解链 <a id='calibration:sec:intro'></a>

本章围绕四个问题展开：

- 如何把校准写成一个显式的最小二乘问题；
- 为什么 1GC 能改正大部分方向无关系统误差；
- 为什么 2GC 必须与成像和天空模型迭代地耦合；
- 当主波束、指向误差、电离层等方向依赖效应出现时，为什么需要 3GC。

与旧版讲义不同，这一章会尽量把“方程形式、求解策略、失败模式、实践流程”放在一起。也就是说，我们不只讨论公式，也讨论什么时候它们会失灵。


In [ ]:
times = np.linspace(0.0, 6.0, 240)

amp_1 = 1.00 + 0.05 * np.sin(2.0 * np.pi * times / 3.1)
phase_1 = 0.35 * np.sin(2.0 * np.pi * times / 2.6 + 0.2)
gain_1 = amp_1 * np.exp(1j * phase_1)

amp_2 = 0.95 + 0.08 * np.cos(2.0 * np.pi * times / 4.0 - 0.4)
phase_2 = -0.28 * np.cos(2.0 * np.pi * times / 3.3 + 0.3)
gain_2 = amp_2 * np.exp(1j * phase_2)

true_vis = 1.2 * np.exp(1j * 0.18)
noise = 0.02 * (RNG.normal(size=times.size) + 1j * RNG.normal(size=times.size))
observed_vis = gain_1 * np.conj(gain_2) * true_vis + noise
calibrated_vis = observed_vis / (gain_1 * np.conj(gain_2))

amp_rms_before = np.sqrt(np.mean((np.abs(observed_vis) - np.abs(true_vis)) ** 2))
amp_rms_after = np.sqrt(np.mean((np.abs(calibrated_vis) - np.abs(true_vis)) ** 2))
phase_rms_before = np.sqrt(
    np.mean((np.angle(observed_vis / true_vis)) ** 2)
)
phase_rms_after = np.sqrt(
    np.mean((np.angle(calibrated_vis / true_vis)) ** 2)
)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(times, np.abs(observed_vis), lw=1.5, color="tab:red", label="受污染振幅")
axes[0].plot(times, np.abs(calibrated_vis), lw=1.8, color="tab:blue", label="改正后振幅")
axes[0].axhline(np.abs(true_vis), ls="--", color="black", label="真实振幅")
axes[0].set_ylabel("amplitude [arb.]")
axes[0].set_title("Calibration removes time-dependent amplitude and phase corruption")
axes[0].legend(loc="upper right")

axes[1].plot(times, np.angle(observed_vis / true_vis), lw=1.5, color="tab:red", label="改正前相位误差")
axes[1].plot(times, np.angle(calibrated_vis / true_vis), lw=1.8, color="tab:blue", label="改正后相位误差")
axes[1].axhline(0.0, ls="--", color="black")
axes[1].set_xlabel("time [hour]")
axes[1].set_ylabel("phase error [rad]")
axes[1].legend(loc="upper right")

plt.tight_layout()
print(f"振幅 RMS：改正前 {amp_rms_before:.3f}，改正后 {amp_rms_after:.3f}")
print(f"相位 RMS：改正前 {phase_rms_before:.3f} rad，改正后 {phase_rms_after:.3f} rad")


上面的例子只展示了最理想的情形：我们“知道”两面天线的复增益，因此能把基线上的污染完全除掉。真实问题当然没有这么简单。我们需要面对至少三类不完美：

- **噪声与有限信噪比**：弱校准源会让解本身变得不稳定；
- **天空模型误差**：若模型漏掉了重要结构，求解器可能把天空误差错误地吸收到增益里；
- **误差模型不匹配**：若真实污染是方向依赖的、而我们却只拟合方向无关增益，那么“解得很好”的结果也可能是物理上错误的。

因此，校准从来不是单纯的“求一个复数”，而是一个把观测设计、硬件误差、先验模型和成像结果联系在一起的闭环过程。


#### 本章内容

- [8.1 作为最小二乘问题的校准](8_1_calibration_least_squares_problem.ipynb)：把天线增益求解显式写成非线性最小二乘问题；
- [8.2 第一代校准（1GC）](8_2_1gc.ipynb)：理解校准源、闭合量、增益转移和实际观测链；
- [8.3 第二代校准（2GC）](8_3_2gc.ipynb)：讨论自校准与天空模型之间的迭代耦合；
- [8.4 第三代校准（3GC）](8_4_3gc.ipynb)：讨论方向依赖校准、差分增益和宽场处理的必要性；
- [8.5 延伸阅读与参考文献](8_5_further_reading_and_references.ipynb)：给出进一步深入学习校准的经典资料。
- [校准习题](8_problem_set.ipynb)：把最小二乘、参考天线、模型不完整和交替求解器做成可练习任务。

建议的阅读顺序是先完成 8.1 中的求解器直觉，再进入 8.2 至 8.4。这样在第 9 章构建完整的数据处理工作流时，会更容易理解为什么某些步骤必须按固定顺序执行。
